# Libaries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from src.benchmark import (
    BenchmarkConfig,
    PairDatasetBuilder,
    PairwiseElasticityPipeline,
    CrossElasticitySymmetrizer,
    BootstrapSummarizer,
)
from src.dominick import DominickDataLoader
from src.utils import TemporalSplitter, BlockBootstrapSampler

In [2]:

TRAIN_FRAC = 0.8
N_FOLDS = 5
N_BOOTSTRAP = 20
SELECTED_UPCS = [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]
config = BenchmarkConfig()

# Loader

In [3]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")
df = df[df["upc_code"].isin(SELECTED_UPCS)].copy()

pair_builder = PairDatasetBuilder(control_cols=config.control_cols)
pair_df = pair_builder.build(df)

print(f"Dataset: {df.shape}, Pairs: {pair_df.shape}")

Dataset: (73209, 30), Pairs: (225004, 32)


# K-fold

In [5]:
splitter = TemporalSplitter(week_col="week_id")
pipeline = PairwiseElasticityPipeline(config)

fold_results = []
for fold_idx, (train_fold, val_fold) in enumerate(splitter.expanding_splits(pair_df, N_FOLDS)):
    print(f"Fold {fold_idx} of {N_FOLDS}")
    res = pipeline.run(train_fold, val_fold)
    res["fold"] = fold_idx
    fold_results.append(res)

all_folds = pd.concat(fold_results, ignore_index=True)
ok_folds = all_folds[all_folds["status"] == "ok"].copy()
print(f"K-fold raw: {len(ok_folds)} rows")

Fold 0 of 5
Fold 1 of 5
Fold 2 of 5
Fold 3 of 5
Fold 4 of 5
K-fold raw: 3944 rows


# Bootstrap

In [6]:
train_df, val_df = splitter.single_split(pair_df, train_frac=TRAIN_FRAC)
train_weeks = sorted(train_df["week_id"].unique())

sampler = BlockBootstrapSampler(week_col="week_id", block_size=4, rng=np.random.default_rng(42))

bootstrap_results = []
for b in range(N_BOOTSTRAP):
    print(f"Bootstrap {b} of {N_BOOTSTRAP}")
    train_bs = sampler.sample(train_df, train_weeks)
    res = pipeline.run(train_bs, val_df)
    res["bootstrap_run"] = b
    bootstrap_results.append(res)

all_bootstrap = pd.concat(bootstrap_results, ignore_index=True)
ok_bs = all_bootstrap[all_bootstrap["status"] == "ok"].copy()
print(f"Bootstrap raw: {len(ok_bs)} rows")

Bootstrap 0 of 20
Bootstrap 1 of 20
Bootstrap 2 of 20
Bootstrap 3 of 20
Bootstrap 4 of 20
Bootstrap 5 of 20
Bootstrap 6 of 20
Bootstrap 7 of 20
Bootstrap 8 of 20
Bootstrap 9 of 20
Bootstrap 10 of 20
Bootstrap 11 of 20
Bootstrap 12 of 20
Bootstrap 13 of 20
Bootstrap 14 of 20
Bootstrap 15 of 20
Bootstrap 16 of 20
Bootstrap 17 of 20
Bootstrap 18 of 20
Bootstrap 19 of 20
Bootstrap raw: 10880 rows


# Summaries

In [7]:
symmetrizer = CrossElasticitySymmetrizer()
summarizer = BootstrapSummarizer()

kfold_cross_sym = symmetrizer.symmetrize(ok_folds, run_col="fold")
bootstrap_summary = summarizer.summarize_raw(ok_bs)
bootstrap_cross_sym = symmetrizer.symmetrize(ok_bs, run_col="bootstrap_run")
bootstrap_cross_summary = summarizer.summarize_cross(bootstrap_cross_sym)

print(f"K-fold sym: {len(kfold_cross_sym)}")
print(f"Bootstrap summary: {len(bootstrap_summary)}")
print(f"Bootstrap cross summary: {len(bootstrap_cross_summary)}")

K-fold sym: 1972
Bootstrap summary: 544
Bootstrap cross summary: 272


# Display

In [8]:
display(ok_folds.head(10))
display(ok_folds[["own_elasticity", "cross_elasticity", "mae_val", "rmse_val", "r2_val"]].describe())
display(bootstrap_summary.head(10))
display(kfold_cross_sym.head(10))
display(bootstrap_cross_summary.head(10))
display(all_folds["status"].value_counts(dropna=False))
display(all_bootstrap["status"].value_counts(dropna=False))

,store_code,pair_id,upc_i,upc_j,status,n_train,n_val,own_elasticity,own_elasticity_ci_low,own_elasticity_ci_high,own_elasticity_p_value,cross_elasticity,cross_elasticity_ci_low,cross_elasticity_ci_high,cross_elasticity_p_value,mae_val,rmse_val,r2_val,fold
0,5,3410017306.0__7289000011.0,3410017306,7289000011,ok,49,21,4.106132,-6.420383,14.632647,4.445495e-01,-7.627939,-24.380248,9.124370,0.372155,1.291065,1.674757,-6.916222,0
1,5,3410017306.0__7289000011.0,7289000011,3410017306,ok,49,21,8.222317,-23.156471,39.601104,6.075473e-01,-0.493523,-8.192901,7.205856,0.900023,1.039971,1.293768,-3.872342,0
2,8,1820000784.0__3410010505.0,1820000784,3410010505,ok,140,30,-4.142325,-5.972335,-2.312314,9.144133e-06,-1.300767,-2.715340,0.113806,0.071501,0.667487,0.854567,-0.111614,0
3,8,1820000784.0__3410010505.0,3410010505,1820000784,ok,140,30,-5.176662,-7.208852,-3.144471,5.954817e-07,-0.322594,-1.463468,0.818281,0.579442,0.782885,0.939221,-1.195786,0
4,8,1820000784.0__3410017306.0,1820000784,3410017306,ok,142,30,-4.776690,-6.536305,-3.017075,1.034471e-07,0.811288,-1.669465,3.292042,0.521540,0.594284,0.743710,0.158083,0
5,8,1820000784.0__3410017306.0,3410017306,1820000784,ok,142,30,-6.256416,-8.409012,-4.103820,1.222628e-08,0.137269,-1.081549,1.356087,0.825295,0.598531,0.779146,-1.384955,0
6,8,1820000784.0__7289000011.0,1820000784,7289000011,ok,101,20,-3.493284,-5.736007,-1.250561,2.266749e-03,-0.999529,-7.877796,5.878738,0.775785,0.595662,0.770598,0.123116,0
7,8,1820000784.0__7289000011.0,7289000011,1820000784,ok,101,20,-1.897329,-9.597517,5.802858,6.291412e-01,-0.708827,-2.130977,0.713323,0.328627,0.523370,0.639952,0.022538,0
8,8,3410010505.0__3410017306.0,3410010505,3410017306,ok,141,30,-5.750082,-7.820377,-3.679786,5.220171e-08,0.684322,-1.352670,2.721314,0.510253,0.904389,1.067756,-1.837910,0
9,8,3410010505.0__3410017306.0,3410017306,3410010505,ok,141,30,-6.172083,-8.429384,-3.914783,8.364557e-08,-0.789207,-1.930209,0.351794,0.175205,0.482902,0.627527,-0.547061,0


,own_elasticity,cross_elasticity,mae_val,rmse_val,r2_val
count,3944.000000,3944.000000,3944.000000,3944.000000,3944.000000
mean,-3.760403,-0.197875,0.524869,0.652909,-0.559355
std,2.438620,1.447692,0.247630,0.278161,5.313071
min,-13.526105,-15.425894,0.121328,0.151870,-211.250510
25%,-5.303098,-0.701600,0.396473,0.500785,-0.482744
50%,-3.814866,-0.229523,0.487237,0.616125,-0.027512
75%,-2.332335,0.258226,0.587101,0.735471,0.295084
max,20.926926,15.496618,4.921428,5.314794,0.895395


,store_code,pair_id,upc_i,upc_j,own_elasticity_mean,own_elasticity_std,own_elasticity_ci_low,own_elasticity_ci_high,cross_elasticity_mean,cross_elasticity_std,cross_elasticity_ci_low,cross_elasticity_ci_high,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std
0,8,1820000784.0__3410010505.0,1820000784,3410010505,-3.964395,0.680193,-4.921938,-2.639270,-0.376021,0.325806,-0.961702,0.087784,0.524610,0.032588,0.707262,0.036821,-0.038334,0.108684
1,8,1820000784.0__3410010505.0,3410010505,1820000784,-3.426151,0.505120,-4.243194,-2.515982,0.746976,0.546606,-0.273952,1.524663,0.629345,0.067967,0.768118,0.071242,-0.892317,0.352611
2,8,1820000784.0__7289000011.0,1820000784,7289000011,-3.618175,0.925191,-4.972160,-1.997445,-2.743262,1.146252,-4.278193,-0.438170,0.598903,0.040358,0.768747,0.049698,-0.228423,0.158863
3,8,1820000784.0__7289000011.0,7289000011,1820000784,-0.293997,1.954698,-4.048812,2.727934,-0.762013,0.507922,-1.587889,0.110380,0.656350,0.088843,0.823043,0.096157,-0.456103,0.357755
4,8,3410010505.0__7289000011.0,3410010505,7289000011,-2.931868,0.610548,-3.768094,-1.912406,-0.906398,1.003366,-3.277282,0.380037,0.607058,0.071369,0.735456,0.081244,-0.740695,0.400911
5,8,3410010505.0__7289000011.0,7289000011,3410010505,-1.045188,2.319066,-5.245457,2.509903,-0.821285,0.626210,-1.492621,0.384377,0.671893,0.079762,0.836068,0.085923,-0.498203,0.322734
6,9,1820000784.0__3410010505.0,1820000784,3410010505,-4.867760,0.803384,-6.638667,-3.635060,0.555180,0.530717,-0.543625,1.266655,1.416517,0.162801,1.626091,0.178238,-4.965926,1.283685
7,9,1820000784.0__3410010505.0,3410010505,1820000784,-3.035237,0.404587,-3.604666,-2.331240,0.698938,0.509137,-0.146390,1.534889,0.415358,0.041936,0.503438,0.050358,-0.289171,0.263211
8,9,1820000784.0__7289000011.0,1820000784,7289000011,-4.403238,0.818527,-5.868370,-3.142305,-0.504029,1.005135,-2.169225,1.129105,1.313337,0.187167,1.515422,0.204940,-4.212033,1.401959
9,9,1820000784.0__7289000011.0,7289000011,1820000784,-2.832014,2.380764,-6.815087,0.941460,1.035438,0.429916,0.335254,1.756877,0.437672,0.056006,0.560536,0.051642,0.255687,0.135479


,store_code,pair_id,upc_a,upc_b,fold,cross_elasticity_sym,n_directions,avg_p_value,mae_val_mean,rmse_val_mean,r2_val_mean
0,5,3410017306.0__7289000011.0,3410017306,7289000011,0,-4.060731,2,0.636089,1.165518,1.484262,-5.394282
1,5,3410017306.0__7289000011.0,3410017306,7289000011,1,0.643182,2,0.797879,0.868146,1.016298,-1.729044
2,5,3410017306.0__7289000011.0,3410017306,7289000011,2,-0.731816,2,0.479082,0.361326,0.462331,-0.175906
3,8,1820000784.0__3410010505.0,1820000784,3410010505,0,-0.811680,2,0.325472,0.725186,0.896894,-0.653700
4,8,1820000784.0__3410010505.0,1820000784,3410010505,1,-0.011313,2,0.730523,0.577795,0.703418,-0.039486
5,8,1820000784.0__3410010505.0,1820000784,3410010505,2,0.241610,2,0.645852,0.526109,0.654710,-0.368751
6,8,1820000784.0__3410010505.0,1820000784,3410010505,3,0.177049,2,0.306510,0.487388,0.635517,-0.264984
7,8,1820000784.0__3410010505.0,1820000784,3410010505,4,0.212939,2,0.220509,0.569659,0.712947,-0.270805
8,8,1820000784.0__3410017306.0,1820000784,3410017306,0,0.474278,2,0.673418,0.596408,0.761428,-0.613436
9,8,1820000784.0__3410017306.0,1820000784,3410017306,1,0.091038,2,0.803276,0.725241,0.847705,-0.716971


,store_code,pair_id,upc_a,upc_b,cross_elasticity_sym_mean,cross_elasticity_sym_std,cross_elasticity_sym_ci_low,cross_elasticity_sym_ci_high,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std
0,8,1820000784.0__3410010505.0,1820000784,3410010505,0.185477,0.314908,-0.417184,0.630393,0.576978,0.032577,0.737690,0.032506,-0.465325,0.160917
1,8,1820000784.0__7289000011.0,1820000784,7289000011,-1.752637,0.672256,-2.521215,-0.452347,0.627626,0.047909,0.795895,0.052112,-0.342263,0.187117
2,8,3410010505.0__7289000011.0,3410010505,7289000011,-0.863841,0.373746,-1.591320,-0.328556,0.639475,0.059672,0.785762,0.065243,-0.619449,0.285241
3,9,1820000784.0__3410010505.0,1820000784,3410010505,0.627059,0.360136,-0.115955,1.154257,0.915937,0.090558,1.064765,0.100390,-2.627548,0.693560
4,9,1820000784.0__7289000011.0,1820000784,7289000011,0.265705,0.509478,-0.661215,1.125004,0.875505,0.095765,1.037979,0.103781,-1.978173,0.700654
5,9,1820000784.0__8248812345.0,1820000784,8248812345,-0.285995,0.237808,-0.799753,-0.020547,1.003154,0.076199,1.157004,0.081413,-2.245340,0.602574
6,9,3410010505.0__7289000011.0,3410010505,7289000011,-0.453369,0.794500,-1.817329,0.826269,0.448792,0.049760,0.551405,0.047753,0.048650,0.160462
7,9,3410010505.0__8248812345.0,3410010505,8248812345,-0.040230,0.288478,-0.484859,0.354703,0.502838,0.032617,0.617718,0.033263,0.078898,0.109749
8,9,7289000011.0__8248812345.0,7289000011,8248812345,0.552724,0.432273,-0.081905,1.411933,0.545152,0.051888,0.667457,0.055818,0.218372,0.134426
9,12,1820000784.0__3410010505.0,1820000784,3410010505,0.465981,0.261436,0.082432,0.876611,0.480509,0.027656,0.605768,0.027459,-0.054139,0.095339


status
ok    3944
Name: count, dtype: int64

status
ok    10880
Name: count, dtype: int64

# Save

In [9]:
ok_folds.to_csv("../data/benchmark_kfold_raw.csv", index=False)
ok_bs.to_csv("../data/benchmark_bootstrap_raw.csv", index=False)
kfold_cross_sym.to_csv("../data/benchmark_cross_generalization_folds.csv", index=False)
bootstrap_summary.to_csv("../data/benchmark_elasticities_bootstrap_summary.csv", index=False)
bootstrap_cross_sym.to_csv("../data/benchmark_cross_bootstrap_sym_raw.csv", index=False)
bootstrap_cross_summary.to_csv("../data/benchmark_cross_bootstrap_summary.csv", index=False)

print("Saved OK")

Saved OK
